In [ ]:
'''
What is ensemble - Combining multiple models to improve performance
Types of ensemple techniques:
    1 - Bagging - Bootstrap Aggregating - Random sampling with replacement: Example random forest
    2 - Boosting - Sequentially training models to correct errors of previous models: example AdaBoost, Gradient Boosting
    3 - Stacking - Combining predictions from multiple models using a meta-model
    4 - Cascading - Using predictions from one model as input for another model


Random forest - this is just multiple decision trees running in parallel
    - Each tree is trained on a random subset of the data (Bootstrap sampling)
        - Random rows and random columns
        - rows can duplicate (adds to useful diversity), but columns cannot: because duplicate feature increase wont add value but will add to compute
    - Few rows get repeated, few dont and the ones which dont get picked by the tree are called out of bag samples (OOB) for that tree
    - Base learners are validated using oob samples
        - oob score: Basically, average output of a point across all trees that did not see that point during training
    - Hyperparameters of Random Forest:
        - Number of trees (n_estimators)
        - max samples (max_samples) ( ratio as input): The number of samples to draw from the training data to train each tree. If None, then draw all samples.
        - max features (max_features): The number of features to consider when looking for the best split. If None, then consider all features.
        - max depth (max_depth): The maximum depth of the tree. If None, then nodes are expanded until all leaves are pure or until all leaves contain less than min_samples_split samples.
    Regularisation:
        - Pruning (Cost complexity pruning) (ccp_alpha): Cutting down the trees till we dont see any significant drop in accuracy.
        - this is also used as alpha



    Bias - Variance tradeoff:
        - Random Forest reduces variance by averaging multiple trees, however bias may increase slightly due to the randomness introduced in the training process
Error in ml:
    variance**2 + bias**2 + irreducible error

Paralleling computation:
    - Random Forest is inherently parallelizable because each tree can be trained independently on different subsets of the data. This allows for faster training times, especially when using multiple CPU cores or distributed computing environments


Steps of Random Forest:
    1 - Train-val-test split
    2 - Train a random forest model on the training set
    3 - tuning all the parameters except alpha (ccp_alpha) using cross validation
    4 - Tune alpha (ccp_alpha) using cross validation
    5 - Evaluate the final model on the test set
    

Grid search CV: Tunes multiple parameters at once, but is computationally expensive. It exhaustively searches through a specified parameter grid to find the best combination of hyperparameters for a given model. 
                It evaluates each combination using cross-validation and selects the one that yields the best performance based on a chosen metric (e.g., accuracy, F1 score).

Random search CV: Tunes one parameter at a time, but is computationally efficient. It randomly samples a specified number of hyperparameter combinations from a defined search space.

Feature importance is calculated by measuring the decrease in impurity that results from splitting on a particular feature across all trees in the forest.
 Features that lead to larger decreases in impurity are considered more important for making accurate predictions.



'''

# Fetching Data

In [2]:
import pandas as pd
import numpy as np

!gdown 19L3rYatfhbBL1r5MHrv-p_oM2wlvrhqk
!gdown 1OHLKJwA3qZopKPvlKoRldM6BvA1A4dYF
!gdown 1N7O_fWCTJLu8SIa_paKcDEzllgpMk8sK
!gdown 12Bh2AN8LcZAlg20ehpQrEWccUDaSdsOG

Downloading...
From: https://drive.google.com/uc?id=19L3rYatfhbBL1r5MHrv-p_oM2wlvrhqk
To: c:\Users\FNP\Documents\DSML\14_ml_supervised_algorithms\14.5_ml_bagging_and_random_forest\preprocessed_X_sm.pickle

  0%|          | 0.00/534k [00:00<?, ?B/s]
100%|██████████| 534k/534k [00:00<00:00, 11.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1OHLKJwA3qZopKPvlKoRldM6BvA1A4dYF
To: c:\Users\FNP\Documents\DSML\14_ml_supervised_algorithms\14.5_ml_bagging_and_random_forest\X_test.pickle

  0%|          | 0.00/111k [00:00<?, ?B/s]
100%|██████████| 111k/111k [00:00<00:00, 1.35MB/s]
Downloading...
From: https://drive.google.com/uc?id=1N7O_fWCTJLu8SIa_paKcDEzllgpMk8sK
To: c:\Users\FNP\Documents\DSML\14_ml_supervised_algorithms\14.5_ml_bagging_and_random_forest\y_sm.pickle

  0%|          | 0.00/15.4k [00:00<?, ?B/s]
100%|██████████| 15.4k/15.4k [00:00<00:00, 2.37MB/s]
Downloading...
From: https://drive.google.com/uc?id=12Bh2AN8LcZAlg20ehpQrEWccUDaSdsOG
To: c:\Users\FNP\Documents\DSML\14_

In [17]:
import pickle
import pandas as pd
# Load data (deserialize)
with open('preprocessed_X_sm.pickle', 'rb') as handle:
    X_train = pd.read_pickle(handle)

with open('X_test.pickle', 'rb') as handle:
    X_test = pd.read_pickle(handle)

with open('y_sm.pickle', 'rb') as handle:
    y_train = pd.read_pickle(handle)

with open('y_test.pickle', 'rb') as handle:
    y_test = pd.read_pickle(handle)

# Benchmarking using simple decusion tree

In [18]:
from sklearn.tree import DecisionTreeClassifier

model=DecisionTreeClassifier(random_state=7,max_depth=4)
model.fit(X=X_train,y=y_train)

print("train accuracy is ", model.score(X_train,y_train))
print("test accuracy is ", model.score(X_test,y_test))

train accuracy is  0.8376623376623377
test accuracy is  0.7880434782608695


In [19]:
X_train

,Age,DailyRate,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,...,YearsWithCurrManager,BusinessTravel_Non-Travel,BusinessTravel_Travel_Frequently,BusinessTravel_Travel_Rarely,Department_Human Resources,Department_Research & Development,Department_Sales,MaritalStatus_Divorced,MaritalStatus_Married,MaritalStatus_Single
0,36,1174,3,4,0.233871,1,0,99,3,2,...,1,0.000000,0.000000,1.0,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000
1,21,546,5,1,0.127479,3,1,97,3,1,...,2,0.000000,0.000000,1.0,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
2,43,422,1,3,0.151584,4,0,33,3,2,...,2,0.000000,1.000000,0.0,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000
3,42,188,29,3,0.127479,2,1,56,1,2,...,0,0.000000,0.000000,1.0,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
4,35,992,1,3,0.127479,4,1,68,2,1,...,2,0.000000,0.000000,1.0,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1843,41,1077,10,3,0.187471,2,1,85,2,2,...,4,0.407817,0.592183,0.0,0.000000,0.407817,0.592183,0.592183,0.000000,0.407817
1844,55,310,7,1,0.239573,3,1,71,3,1,...,7,0.000000,0.000000,1.0,0.000000,0.998141,0.001859,0.000000,0.998141,0.001859
1845,36,899,5,2,0.143813,1,0,92,2,1,...,2,0.677643,0.322357,0.0,0.000000,1.000000,0.000000,0.677643,0.322357,0.000000
1846,40,292,7,3,0.151584,1,1,84,3,3,...,5,0.192577,0.807423,0.0,0.000000,1.000000,0.000000,0.807423,0.192577,0.000000


# Random Forest code implementation

In [42]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, cross_validate

In [43]:
model=RandomForestClassifier(random_state=7,n_estimators=100,max_depth=4)
model.fit(X=X_train,y=y_train)

print("train accuracy is ", model.score(X_train,y_train))
print("test accuracy is ", model.score(X_test,y_test))


train accuracy is  0.8901515151515151
test accuracy is  0.8369565217391305


## Kfold and cross validation

In [44]:
kf=KFold(n_splits=10)
cv_results=cross_validate(model,X_train,y_train,cv=kf,scoring='accuracy',return_train_score=True)

In [45]:
cv_results

{'fit_time': array([0.19554377, 0.1835084 , 0.1697433 , 0.17453361, 0.17951846,
        0.17362618, 0.16760898, 0.1667912 , 0.18331027, 0.17353702]),
 'score_time': array([0.01089168, 0.00778747, 0.00975108, 0.02296495, 0.00789261,
        0.00684595, 0.0068481 , 0.00978208, 0.02742815, 0.00791407]),
 'test_score': array([0.85945946, 0.8       , 0.82162162, 0.74594595, 0.75675676,
        0.82702703, 0.8972973 , 0.92432432, 0.88043478, 0.91304348]),
 'train_score': array([0.87973542, 0.88274203, 0.89116055, 0.88695129, 0.88334336,
        0.89176188, 0.87733013, 0.88033674, 0.88461538, 0.8828125 ])}

In [46]:
print(f"K-Fold Accuracy Mean: \n Train: {cv_results['train_score'].mean()*100:.2f} \n Validation: {cv_results['test_score'].mean()*100:.2f}")

K-Fold Accuracy Mean: 
 Train: 88.41 
 Validation: 84.26


## Grid Search CV

In [31]:
from sklearn.model_selection import GridSearchCV

In [32]:
params = {
    'n_estimators': [100,200,300,400],
    'max_depth': [3,5,10],
    'criterion': ['gini', 'entropy'],
    'bootstrap': [True, False],
    'max_features': [8,9,10,5]
}

In [38]:
cv=GridSearchCV(estimator=RandomForestClassifier(),param_grid=params,scoring='accuracy',cv=3,n_jobs=1)

In [39]:
cv.fit(X_train,y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestClassifier()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'bootstrap': [True, False], 'criterion': ['gini', 'entropy'], 'max_depth': [3, 5, ...], 'max_features': [8, 9, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose 

In [41]:
print(cv.best_score_)
print(cv.best_params_)

0.9047619047619048
{'bootstrap': True, 'criterion': 'gini', 'max_depth': 10, 'max_features': 5, 'n_estimators': 100}


## Tuning alpha

In [53]:
import numpy as np

In [54]:
list_alphas=np.linspace(0, 0.4, 20)

In [ ]:
kf_2=KFold(n_splits=5)


In [56]:
gridcv=GridSearchCV(estimator=RandomForestClassifier(bootstrap=True,criterion='gini',max_depth=10,max_features=5,n_estimators=100),param_grid={'ccp_alpha':list_alphas},scoring='accuracy',cv=kf_2,n_jobs=1)

In [57]:
gridcv.fit(X_train,y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ax_features=5)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'ccp_alpha': array([0. ..., 0.4 ])}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split...shuffle=False)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbosity of in

In [58]:
print(gridcv.best_score_)
print(gridcv.best_params_)

0.9053746429356186
{'ccp_alpha': np.float64(0.0)}


In [66]:
pd.DataFrame(
    {
        "feature": X_train.columns,
        "importance": gridcv.best_estimator_.feature_importances_,
    }
).sort_values(by="importance", ascending=False)

,feature,importance
35,MaritalStatus_Single,0.077913
10,JobRole,0.063841
4,EducationField,0.053301
19,StockOptionLevel,0.052175
12,MonthlyIncome,0.049464
31,Department_Research & Development,0.045535
34,MaritalStatus_Married,0.041392
5,EnvironmentSatisfaction,0.039318
8,JobInvolvement,0.037970
11,JobSatisfaction,0.037093


## Excercise

In [2]:
import numpy as np

In [3]:
n_estimators = [int(x) for x in np.linspace(start = 200, stop = 2000, num = 2)]
max_depth = [int(x) for x in np.linspace(10, 110, num = 2)]


In [4]:
hyper=[]

for i in n_estimators:
    for j in max_depth:
        hyper.append([i,j])


In [8]:
hyper[0][1]

10